# 02 · Clean pages lab (ScanTailor)

Tune the imaging front-end: ScanTailor splits spreads, deskews, crops the
scanner bed, normalises illumination and (in `mixed` mode) binarises text while
keeping picture tones. Cleaned pages are the input to OCR + MRC rendering.

In [ ]:
import sys
from pathlib import Path

# project importable when the kernel isn't the project .venv
sys.path.insert(0, str(Path.cwd().parent))

from evilflowers_books_digitalizer.runtime import load_runtime
from evilflowers_books_digitalizer.cache import LocalCache

# ONE source of truth for cache_dir + output_dir (paths resolve to the project
# root regardless of the notebook's working directory) — this is the unified
# cache every notebook and the production runner share.
rt = load_runtime()
cache = LocalCache(rt.cache_dir)
print("cache :", rt.cache_dir)
print("output:", rt.output_dir)

In [ ]:
import pymupdf

def show_pdf_page(pdf_path, page=1, dpi=110):
    """Render one PDF page inline (for visual QA)."""
    from IPython.display import Image, display
    d = pymupdf.open(str(pdf_path))
    page = min(page, len(d) - 1)
    display(Image(data=d[page].get_pixmap(dpi=dpi).tobytes("png")))

def page_render_ms(pdf_path, dpi=150):
    """Mean per-page render time (ms) — the viewer-load-speed proxy."""
    import time
    d = pymupdf.open(str(pdf_path))
    ts = []
    for i in range(len(d)):
        t = time.time(); d[i].get_pixmap(dpi=dpi); ts.append((time.time() - t) * 1000)
    return sum(ts) / len(ts)

def cached_books():
    """(source, book_id) for every book staged in the local cache."""
    scans = rt.cache_dir / "scans"
    return sorted((p.parent.name, p.name) for p in scans.glob("*/*") if p.is_dir())

### Run ScanTailor on a sample of one cached book

In [ ]:
import shutil, cv2
from evilflowers_books_digitalizer.pipeline.base import BookContext
from evilflowers_books_digitalizer.pipeline.steps.scantailor import ScanTailorScans

src, bid = cached_books()[0]
frames = cache.list_tiffs(src, bid)
# a readable spread across the book (skip unreadable/corrupt frames)
idx = [i for i in (1, len(frames)//3, len(frames)//2, 2*len(frames)//3) if i < len(frames)]
sample = [frames[i] for i in idx if cv2.imread(str(frames[i])) is not None]

work = rt.cache_dir / "lab" / "clean_work"
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
ctx = BookContext(source=src, book_id="LAB", work_dir=work,
                  output_dir=rt.cache_dir / "lab" / "clean_out", tiffs=sample)
ctx = ScanTailorScans(color_mode="mixed", dpi=300, output_dpi=600).run(ctx)
print("cleaned", len(ctx.tiffs), "pages")

### Before / after

In [ ]:
import cv2
from IPython.display import Image, display
raw = cv2.imread(str(sample[0]))
clean = cv2.imread(str(sorted(ctx.tiffs)[0]))
for label, im in [("RAW", raw), ("CLEAN", clean)]:
    print(label, im.shape)
    s = cv2.resize(im, (im.shape[1]//4, im.shape[0]//4))
    display(Image(data=cv2.imencode(".png", s)[1].tobytes()))

### Knobs worth tuning (`[scantailor]` in `pipeline.toml`)

| key | effect |
|---|---|
| `color_mode` | `mixed` (crisp text, kills bleed-through) vs `color_grayscale` (keep tones, pair with DocRes) |
| `output_dpi` | 600 from 300-dpi input = supersampled, smoother glyph edges |
| `margins_mm` | white margin added around content |
| `despeckle` | `cautious` / `normal` / `aggressive` |
| `normalize_illumination` | flatten grey backgrounds |

Re-run the cell above after changing arguments to compare.